# Silver Layer - Data Transformation

## Import

In [8]:
import pandas as pd
from datetime import datetime, timezone
import os

from data_quality import (
    remove_duplicates,
    validate_unique,
    validate_not_null,
    validate_timestamp_order
)

## Variables

In [9]:
datasets = {
    "communications": {
        "bronze_path": "../../data/bronze/communications",
        "rename_columns": {
            "broker_comany_id": "broker_company_id"
        },
        "data_types": {
            "external_id": "string",
            "carrier_company_id": "string",
            "broker_company_id": "string",
            "direction": "string",
            "channel": "string",
            "status": "string",
            "created_at": "datetime",
            "updated_at": "datetime",
            "from_contact_type": "string",
            "to_contact_type": "string",
            "thread_id": "string"
        },
        "remove_duplicates": True,
        "unique_columns": [],
        "not_null_columns": [
            "external_id",
            "broker_company_id",
            "direction",
            "channel",
            "status",
            "created_at",
            "updated_at"
        ],
        "timestamp_order": [
            ("created_at", "updated_at")
        ]
    },

    "brokers": {
        "bronze_path": "../../data/bronze/brokers",
        "rename_columns": {
            "_c0": "id",
            "_c1": "name"
        },
        "data_types": {
            "id": "string",
            "name": "string"
        },
        "remove_duplicates": True,
        "unique_columns": [
            "id"
        ],
        "not_null_columns": [
            "id",
            "name"
        ],
        "timestamp_order": []
    },

    "carriers": {
        "bronze_path": "../../data/bronze/carriers",
        "rename_columns": {},
        "data_types": {
            "id": "string",
            "name": "string"
        },
        "remove_duplicates": True,
        "unique_columns": [
            "id"
        ],
        "not_null_columns": [
            "id",
            "name"
        ],
        "timestamp_order": []
    }
}

## Functions

In [10]:
def rename_columns(df, rename_map):
    return df.rename(columns=rename_map)

In [11]:
def cast_columns(df, cast_types):

    df = df.copy()

    for column, data_type in cast_types.items():

        if data_type == "datetime":
            df[column] = pd.to_datetime(
                df[column],
                utc=True,
                errors="raise"
            )

        else:
            df[column] = df[column].astype(data_type)

    return df

## Main

In [12]:
transformed_datasets = {}

for dataset_name, config in datasets.items():

    print(f"Reading Bronze Layer '{dataset_name}'...")

    df = pd.read_parquet(
        config["bronze_path"]
    )

    print(f"Renaming Columns '{dataset_name}'...")

    df = rename_columns(
        df,
        config["rename_columns"]
    )

    print(f"Casting Columns '{dataset_name}'...")

    df = cast_columns(
        df,
        config["data_types"]
    )

    # Cleaning

    if config.get("remove_duplicates", False):

        print(f"Removing Duplicates '{dataset_name}'...")

        df = remove_duplicates(df)

    # Data Quality Validations

    if config.get("unique_columns"):

        print(f"Validating Unique Columns '{dataset_name}'...")

        validate_unique(
            df,
            config["unique_columns"],
            dataset_name
        )

    if config.get("not_null_columns"):

        print(f"Validating Not Null Columns '{dataset_name}'...")

        validate_not_null(
            df,
            config["not_null_columns"],
            dataset_name
        )

    if config.get("timestamp_order"):

        print(f"Validating Timestamp Order '{dataset_name}'...")

        validate_timestamp_order(
            df,
            config["timestamp_order"],
            dataset_name
        )

    # Silver metadata

    df["_transformed_at"] = datetime.now(timezone.utc)

    transformed_datasets[dataset_name] = df

    print(
        f"'{dataset_name}' transformed and validated successfully."
    )

Reading Bronze Layer 'communications'...


Renaming Columns 'communications'...
Casting Columns 'communications'...
Removing Duplicates 'communications'...
Validating Not Null Columns 'communications'...
Validating Timestamp Order 'communications'...
'communications' transformed and validated successfully.
Reading Bronze Layer 'brokers'...
Renaming Columns 'brokers'...
Casting Columns 'brokers'...
Removing Duplicates 'brokers'...
Validating Unique Columns 'brokers'...
Validating Not Null Columns 'brokers'...
'brokers' transformed and validated successfully.
Reading Bronze Layer 'carriers'...
Renaming Columns 'carriers'...
Casting Columns 'carriers'...
Removing Duplicates 'carriers'...
Validating Unique Columns 'carriers'...
Validating Not Null Columns 'carriers'...
'carriers' transformed and validated successfully.


In [13]:
# Save Silver only after every dataset passes validation

for dataset_name, df in transformed_datasets.items():

    output_path = f"../../data/silver/{dataset_name}"

    # Production S3 example:
    # output_path = f"s3://freighthero-data/silver/{dataset_name}"

    os.makedirs(
        output_path,
        exist_ok=True
    )

    df.to_parquet(
        f"{output_path}/data.parquet",
        index=False
    )

    print(
        f"Silver dataset '{dataset_name}' saved successfully."
    )

Silver dataset 'communications' saved successfully.
Silver dataset 'brokers' saved successfully.
Silver dataset 'carriers' saved successfully.
